# Fine-tuning LLaMA 3.2-1B with LoRA/PEFT on SciQ

This notebook fine-tunes **LLaMA 3.2-1B** using **LoRA (Low-Rank Adaptation)** via the PEFT library on the [SciQ dataset](https://huggingface.co/datasets/allenai/sciq).

**Task:** Given a context paragraph and a question, generate the correct answer.

**Approach:**
- Parameter-Efficient Fine-Tuning (PEFT) with LoRA — only ~0.1% of parameters are trained
- LoRA applied to `q_proj` and `v_proj` attention layers
- Early stopping to prevent overfitting
- Validation loss logged after every epoch

---
### Requirements
```
pip install torch transformers datasets peft accelerate pandas pyarrow rouge-score sentence-transformers
```

## 1. Install Dependencies

In [ ]:

!pip install torch transformers datasets peft accelerate pandas pyarrow rouge-score sentence-transformers

## 2. Imports

In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType

## 3. Configuration

All key parameters are defined here. Adjust `LEARNING_RATE`, `NUM_EPOCHS`, and `OUTPUT_DIR` as needed.

In [ ]:
#  Model & Data
LLM_PATH   = "meta-llama/Llama-3.2-1B" 
OUTPUT_DIR = "results/finetune"
METRICS_PATH = f"{OUTPUT_DIR}/epoch_metrics.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Hyperparameters
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5
BATCH_SIZE    = 4    
MAX_LENGTH    = 512

# LoRA Configuration
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.1

print("=" * 50)
print(f"Model         : {LLM_PATH}")
print(f"Learning Rate : {LEARNING_RATE}")
print(f"Epochs        : {NUM_EPOCHS}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"LoRA r        : {LORA_R}")
print(f"LoRA alpha    : {LORA_ALPHA}")
print(f"Output dir    : {OUTPUT_DIR}")
print("=" * 50)

## 4. Load Dataset

We use the [SciQ dataset](https://huggingface.co/datasets/allenai/sciq) — a science question answering dataset with context paragraphs. It contains ~11,679 training, 1,000 validation, and 1,000 test questions.

In [ ]:
# Load directly from HuggingFace
dataset = load_dataset("allenai/sciq")

def to_dataframe(split):
    df = pd.DataFrame(dataset[split])
    df = df.rename(columns={
        "support":        "Context",
        "question":       "Question",
        "correct_answer": "Answer"
    })
    df = df.dropna(subset=["Context", "Question", "Answer"])
    # Drop rows with empty context (no support)
    df = df[df["Context"].str.strip() != ""]
    return df[["Context", "Question", "Answer"]]

train_df = to_dataframe("train")
val_df   = to_dataframe("validation")
test_df  = to_dataframe("test")

print(f"Train samples      : {len(train_df)}")
print(f"Validation samples : {len(val_df)}")
print(f"Test samples       : {len(test_df)}")

## 5. Load Tokenizer

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LLM_PATH)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded!")

## 6. Prompt Format & Tokenization

We format each sample as a prompt that gives the model the context and question, then trains it to predict only the answer tokens.

**Label masking:** The prompt part (context + question) is masked with `-100` so the loss is only computed on the answer tokens. This prevents the model from just memorizing the prompt format.

In [ ]:
def make_prompt(context, question, answer=None):
    prompt = f"""Answer the question using ONLY the context below.
Give only the final answer.

Context:
{context}

Question:
{question}

Answer:
"""
    if answer is not None:
        prompt += answer + tokenizer.eos_token
    return prompt

In [ ]:
def tokenize(examples):
    prompts = [
        make_prompt(c, q, a)
        for c, q, a in zip(examples["Context"], examples["Question"], examples["Answer"])
    ]

    tokenized = tokenizer(
        prompts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

    input_ids = [ids[:MAX_LENGTH] for ids in tokenized["input_ids"]]
    labels    = [ids.copy() for ids in input_ids]

    # Mask prompt tokens — only compute loss on answer tokens
    for i, prompt in enumerate(prompts):
        answer_start = prompt.rfind("Answer:")
        prefix       = prompt[:answer_start + len("Answer:")]
        prefix_ids   = tokenizer(prefix, add_special_tokens=False)["input_ids"]
        prefix_len   = min(len(prefix_ids), MAX_LENGTH)
        labels[i][:prefix_len] = [-100] * prefix_len

    tokenized["input_ids"] = input_ids
    tokenized["labels"]    = labels
    return tokenized

print("Tokenizing datasets...")
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(tokenize,   batched=True, remove_columns=val_dataset.column_names)

print(f"Train tokenized : {len(train_dataset)} samples")
print(f"Val tokenized   : {len(val_dataset)} samples")

## 7. Training Arguments

Key choices:
- `eval_strategy="epoch"` — evaluate on validation set after every epoch
- `load_best_model_at_end=True` — automatically keep the best checkpoint
- `EarlyStoppingCallback(patience=2)` — stop if validation loss doesn't improve for 2 epochs

In [ ]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LEARNING_RATE,
    weight_decay                = 0.01,
    fp16                        = torch.cuda.is_available(),  # only if GPU available
    warmup_steps                = 100,
    lr_scheduler_type           = "cosine",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    logging_steps               = 10,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    save_total_limit            = 2,
)
print("Training arguments set!")

## 8. Load Model & Apply LoRA

We load the base LLaMA model and wrap it with LoRA adapters. Only the LoRA parameters (~0.1% of total) are trained — the base model weights are frozen.

In [ ]:
print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    LLM_PATH,
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map  = "auto"
)
model.config.use_cache = False
print("Base model loaded!")

In [ ]:
print("Applying LoRA...")
lora_config = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    target_modules = ["q_proj", "v_proj"],  # attention projection layers
    lora_dropout   = LORA_DROPOUT,
    bias           = "none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 9. Train

The callback logs `train_loss` and `eval_loss` after every epoch and saves them to `epoch_metrics.csv`.

In [ ]:
epoch_metrics_log = []

class EpochMetricsCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        epoch      = round(state.epoch)
        train_loss = next(
            (x["loss"] for x in reversed(state.log_history) if "loss" in x), None
        )
        eval_loss = metrics.get("eval_loss", None)

        print(f"\n{'='*40}")
        print(f"Epoch {epoch} complete")
        print(f"  Train loss : {train_loss:.4f}" if train_loss else "  Train loss : N/A")
        print(f"  Val loss   : {eval_loss:.4f}"  if eval_loss  else "  Val loss   : N/A")
        print(f"{'='*40}")

        epoch_metrics_log.append({
            "epoch":      epoch,
            "train_loss": round(train_loss, 6) if train_loss else None,
            "eval_loss":  round(eval_loss,  6) if eval_loss  else None,
            "lr":         LEARNING_RATE
        })
        pd.DataFrame(epoch_metrics_log).to_csv(METRICS_PATH, index=False)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    callbacks     = [
        EpochMetricsCallback(),
        EarlyStoppingCallback(early_stopping_patience=2)
    ]
)

print("Starting training...")
trainer.train()
print("Training complete!")

## 10. Save Best Model

In [ ]:
best_model_path = f"{OUTPUT_DIR}/best_model"
model.save_pretrained(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f"Best model (LoRA weights) saved, {best_model_path}")

## 11. Training Summary

In [ ]:
metrics_df = pd.DataFrame(epoch_metrics_log)
best_epoch = metrics_df.loc[metrics_df["eval_loss"].idxmin(), "epoch"]
best_loss  = metrics_df["eval_loss"].min()

print("Training Summary")
print(metrics_df.to_string(index=False))
print(f"\nBest epoch    : {best_epoch}")
print(f"Best val loss : {best_loss:.4f}")

metrics_df

## 12. Evaluation on Test Set

Now we evaluate the fine-tuned model on the test set and compute ROUGE-1, F1, and Semantic Similarity.

In [ ]:
import re
from peft import PeftModel
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util
from collections import Counter

# Load model for inference
print("Loading fine-tuned model for evaluation...")
base_model = AutoModelForCausalLM.from_pretrained(
    LLM_PATH,
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map  = "auto"
)
eval_model = PeftModel.from_pretrained(base_model, best_model_path)
eval_model.eval()
print("Model ready for evaluation!")

In [ ]:
def make_inference_prompt(context, question):
    return f"""Answer the question using ONLY the context below.
Give only the final answer. Do not guess.

Context:
{context}

Question:
{question}

Answer:
"""

def generate_answer(context, question):
    prompt  = make_inference_prompt(context, question)
    inputs  = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(eval_model.device)
    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens     = 10,
            do_sample          = False,
            repetition_penalty = 1.05,
            pad_token_id       = tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    match   = re.search(r"Answer:\s*(.*)", decoded)
    answer  = match.group(1).strip() if match else ""
    return answer.split("\n")[0]

# Run inference
print("Running inference on test set...")
results = []
for i, row in enumerate(test_df.itertuples()):
    predicted = generate_answer(row.Context, row.Question)
    results.append({
        "Question":         row.Question,
        "Context":          row.Context,
        "Predicted_Answer": predicted,
        "Answer":           row.Answer
    })

print("Inference complete!")
results_df = pd.DataFrame(results)

In [ ]:
def normalize(text):
    if text is None or isinstance(text, float): return ""
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return re.sub(r"\s+", " ", text).strip()

results_df["pred_norm"] = results_df["Predicted_Answer"].apply(normalize)
results_df["gold_norm"] = results_df["Answer"].apply(normalize)

# ROUGE-1
rouge  = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
results_df["rouge1"] = results_df.apply(
    lambda x: rouge.score(x["gold_norm"], x["pred_norm"])["rouge1"].fmeasure, axis=1
)
rouge1_score = results_df["rouge1"].mean()

# F1
def f1_score_gen(pred, gold):
    p_tok, g_tok = pred.split(), gold.split()
    if not p_tok or not g_tok: return int(p_tok == g_tok)
    common = Counter(p_tok) & Counter(g_tok)
    nc = sum(common.values())
    if nc == 0: return 0.0
    p, r = nc / len(p_tok), nc / len(g_tok)
    return 2 * p * r / (p + r)

results_df["f1"] = results_df.apply(
    lambda x: f1_score_gen(x["pred_norm"], x["gold_norm"]), axis=1
)
f1_score = results_df["f1"].mean()

# Semantic Similarity
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
pred_emb = embedder.encode(results_df["Predicted_Answer"].tolist(), convert_to_tensor=True)
gold_emb = embedder.encode(results_df["Answer"].tolist(),           convert_to_tensor=True)
cos_sim  = util.cos_sim(pred_emb, gold_emb).diagonal().cpu().numpy()
results_df["semantic_similarity"] = cos_sim
semantic_score = results_df["semantic_similarity"].mean()

print("\n Evaluation Results (Test Set)")
print(f"ROUGE-1            : {rouge1_score:.4f}")
print(f"F1                 : {f1_score:.4f}")
print(f"Semantic Similarity: {semantic_score:.4f}")

# Save results
results_df.to_csv(f"{OUTPUT_DIR}/finetune_final_results.csv", index=False)
pd.DataFrame([{
    "ROUGE1": round(rouge1_score, 7),
    "F1": round(f1_score, 7),
    "Semantic_Similarity": round(float(semantic_score), 7)
}]).to_csv(f"{OUTPUT_DIR}/finetune_evaluation_metrics.csv", index=False)
print(f"\nResults saved to {OUTPUT_DIR}/")